---
**The following expands the analysis to try to find the range of returns possible to achieve in the market with a 
box spread.** 

In [2]:
from pathlib import Path
from datetime import datetime, date
import yfinance as yf
import pandas as pd
import numpy as np

## Do not run the following two code cells if you want to replicate the results
They will give you current data and hence not the data I used to get to the results of the project

## European-Style Index Options: Underlyings and Rationale

Below are the tickers used, the indexes they represent, and confirmation that their options are European-style (exercise only at expiration).

| Ticker  | Index                                          | Option Style   |
|:-------:|:-----------------------------------------------|:--------------:|
| **^VIX** | CBOE Volatility Index                         | European       |
| **^NDX** | NASDAQ-100 Index                              | European       |
| **^RUT** | Russell 2000 Index                            | European       |
| **^XSP** | Mini-S&P 500 Index (scaled SPX)               | European       |
| **^OEX** | S&P 100 Index                                 | European       |
| **^SPX** | S&P 500 Index                                 | European       |
| **^DJX** | Dow Jones Industrial Average Index            | European       |

> **Why European-Style Matters**  
> - These are **index** symbols, not ETFs.  Their options can **only** be exercised on the expiration date, so the payoff of a box spread is truly **locked in**.  
> - If one uses **American-style** options (e.g. SPY, QQQ, DIA, IWM), the short-put leg can be **exercised early** (and calls too around dividends), breaking the synthetic bond and destroying the box-spread arbitrage.  
> - To construct a **risk-free synthetic bond** via a box spread, it is **crucial** to use European-style contracts so that **no** early exercise or assignment risk exists before expiry.


In [ ]:
# --------------------------------------------------------------------
# Box‑spread scanner
# --------------------------------------------------------------------


# =============== SETTINGS ==========================================
TICKERS     = ["^VIX", "^NDX", "^RUT", "^XSP", "^RUT", "^OEX", "^SPX", "^DJX"]   
MAX_DAYS    = 120      # only expiries within this many calendar days
MIN_OI      = 50       # minimum open interest to count as liquid
MIN_VOL     = 10       # minimum volume to count as liquid
MIN_BID     = 0.05     # ignore quotes with tiny bid
MIN_ASK     = 0.05     # ignore quotes with tiny ask
MAX_SPREAD  = 0.10     # max allowed (ask - bid)

DATA_DIR = r""  #if you want to run this code, you have to input your working directory here 
Path(DATA_DIR).mkdir(parents=True, exist_ok=True)
# ===================================================================


def fetch_chain(ticker: str, expiry: str) -> pd.DataFrame:
    t = yf.Ticker(ticker)
    calls, puts = t.option_chain(expiry).calls, t.option_chain(expiry).puts

    calls = calls.copy(); puts = puts.copy()
    calls["type"] = "call"; puts["type"] = "put"
    df = pd.concat([calls, puts], ignore_index=True)
    df["underlying"] = ticker
    df["expiry"]     = expiry
    df["days2exp"]   = (datetime.strptime(expiry, "%Y-%m-%d").date()
                       - date.today()).days

    df.to_csv(f"{DATA_DIR}/{ticker}_{expiry}.csv", index=False)
    return df

new_all_boxes = []

for ticker in TICKERS:
    tk = yf.Ticker(ticker)

    for expiry in tk.options:
        days = (datetime.strptime(expiry, "%Y-%m-%d").date()
                - date.today()).days
        if days <= 0 or days > MAX_DAYS:
            continue

        chain = fetch_chain(ticker, expiry)

        # ---- enriched liquidity filter ----
        calls = chain[
            (chain.type == "call") &
            (chain.openInterest >= MIN_OI) &
            (chain.volume        >= MIN_VOL) &
            (chain.bid           >= MIN_BID) &
            (chain.ask           >= MIN_ASK) &
            ((chain.ask - chain.bid) <= MAX_SPREAD)
        ]
        puts = chain[
            (chain.type == "put") &
            (chain.openInterest >= MIN_OI) &
            (chain.volume        >= MIN_VOL) &
            (chain.bid           >= MIN_BID) &
            (chain.ask           >= MIN_ASK) &
            ((chain.ask - chain.bid) <= MAX_SPREAD)
        ]

        common_strikes = sorted(set(calls.strike) & set(puts.strike))
        if len(common_strikes) < 2:
            continue

        # --------------------------------------------------------------
        # Build every bull‑call + bear‑put box spread  (K1 < K2)
        # --------------------------------------------------------------
        for i, K1 in enumerate(common_strikes):
            for K2 in common_strikes[i+1:]:
                long_call  = calls .loc[calls .strike == K1].iloc[0]
                short_call = calls .loc[calls .strike == K2].iloc[0]
                short_put  = puts  .loc[puts  .strike == K1].iloc[0]
                long_put   = puts  .loc[puts  .strike == K2].iloc[0]

                entry_cost = (
                    long_call.ask
                  - short_call.bid
                  - short_put.bid
                  + long_put.ask
                ) * 100.0

                payoff  = (K2 - K1) * 100.0
                profit  = payoff - entry_cost
                raw_ret = profit / abs(entry_cost) if entry_cost != 0 else np.nan
                T_year  = days / 365.0
                ann_yld = raw_ret / T_year if T_year>0 and not np.isnan(raw_ret) else np.nan

                new_all_boxes.append({
                    "underlying": ticker,
                    "expiry":     expiry,
                    "days2exp":   days,
                    "K1":         K1,
                    "K2":         K2,
                    "entry_cost": entry_cost,
                    "payoff":     payoff,
                    "profit":     profit,
                    "raw_return": raw_ret,
                    "ann_yield":  ann_yld
                })

---
All options prices are saved here to csv such that the results can be re-creatable.

In [ ]:
# ----------------------------- RESULTS --------------------------------
boxes = pd.DataFrame(new_all_boxes)
boxes.to_csv("new_all_box_spreads.csv", index=False)   
boxes

## Do run following code
from this point on, you can run the code for replicability 

In [5]:
boxes = pd.read_csv("new_all_box_spreads.csv")

best10  = boxes.sort_values("ann_yield", ascending=False).head(10)
worst10 = boxes.sort_values("ann_yield", ascending=True ).head(10)

print("=== TOP 10 annualised yields ===")
display(best10[["underlying","expiry","K1","K2", "days2exp", "ann_yield", "raw_return"]])

print("\n=== BOTTOM 10 annualised yields ===")
display(worst10[["underlying","expiry","K1","K2", "days2exp", "ann_yield", "raw_return"]])

print(boxes['ann_yield'].describe())
boxes

=== TOP 10 annualised yields ===


,underlying,expiry,K1,K2,days2exp,ann_yield,raw_return
58,^VIX,2025-06-18,16.0,100.0,55,0.036542,0.005506
63,^VIX,2025-06-18,30.0,100.0,55,0.035264,0.005314
64,^VIX,2025-06-18,31.0,100.0,55,0.031892,0.004806
61,^VIX,2025-06-18,28.0,100.0,55,0.029627,0.004464
65,^VIX,2025-08-20,20.0,40.0,118,0.023374,0.007557
55,^VIX,2025-06-18,16.0,28.0,55,-0.005526,-0.000833
57,^VIX,2025-06-18,16.0,31.0,55,-0.013246,-0.001996
56,^VIX,2025-06-18,16.0,30.0,55,-0.023617,-0.003559
8,^VIX,2025-05-21,19.0,39.0,27,-0.053859,-0.003984
32,^VIX,2025-05-21,25.0,39.0,27,-0.057689,-0.004267



=== BOTTOM 10 annualised yields ===


,underlying,expiry,K1,K2,days2exp,ann_yield,raw_return
70,^XSP,2025-04-25,541.0,542.0,1,-55.677966,-0.152542
66,^XSP,2025-04-25,540.0,541.0,1,-47.608696,-0.130435
67,^XSP,2025-04-25,540.0,542.0,1,-34.683258,-0.095023
75,^XSP,2025-04-25,545.0,546.0,1,-33.181818,-0.090909
73,^XSP,2025-04-25,542.0,545.0,1,-28.076923,-0.076923
71,^XSP,2025-04-25,541.0,545.0,1,-27.037037,-0.074074
68,^XSP,2025-04-25,540.0,545.0,1,-23.878505,-0.065421
72,^XSP,2025-04-25,541.0,546.0,1,-20.660377,-0.056604
74,^XSP,2025-04-25,542.0,546.0,1,-19.846336,-0.054374
69,^XSP,2025-04-25,540.0,546.0,1,-19.028436,-0.052133


count    76.000000
mean     -4.418510
std      11.245291
min     -55.677966
25%      -0.704757
50%      -0.267191
75%      -0.122066
max       0.036542
Name: ann_yield, dtype: float64


,underlying,expiry,days2exp,K1,K2,entry_cost,payoff,profit,raw_return,ann_yield
0,^VIX,2025-05-21,27,19.0,21.0,210.0,200.0,-10.0,-0.047619,-0.643739
1,^VIX,2025-05-21,27,19.0,24.0,511.0,500.0,-11.0,-0.021526,-0.291005
2,^VIX,2025-05-21,27,19.0,25.0,616.0,600.0,-16.0,-0.025974,-0.351130
3,^VIX,2025-05-21,27,19.0,26.0,711.0,700.0,-11.0,-0.015471,-0.209147
4,^VIX,2025-05-21,27,19.0,27.0,813.0,800.0,-13.0,-0.015990,-0.216163
...,...,...,...,...,...,...,...,...,...,...
71,^XSP,2025-04-25,1,541.0,545.0,432.0,400.0,-32.0,-0.074074,-27.037037
72,^XSP,2025-04-25,1,541.0,546.0,530.0,500.0,-30.0,-0.056604,-20.660377
73,^XSP,2025-04-25,1,542.0,545.0,325.0,300.0,-25.0,-0.076923,-28.076923
74,^XSP,2025-04-25,1,542.0,546.0,423.0,400.0,-23.0,-0.054374,-19.846336


---
possible returns. 

In [4]:
-0.052133/(1/365)

-19.028545


| Scenario | Raw Return | Annualised Yield |
|:--------:|:----------:|:----------------:|
| **Top 10**  | -0.43 % to 0.55 %  | -5.77 % to 3.65 %  |
| **Bottom 10** | –5.21 % to –15.25 % | –1902.84 % to -5567.80 % |


**Short expiries amplify small raw returns**  
   A raw_return of, say, **-15.25%** (15.25 ¢ on $1) over 1 day annualises to

   
   $$
     \frac{-0.052133}{1/365} = -19.028545 \approx-1902.85\%\,.  
   $$  
   Thus even tiny profits/losses look huge when blown up to a full year.

**Raw returns remain modest**  
   Most of these boxes deliver raw gains or losses of only a few percent—well within normal transaction‐cost noise.

**Overly negative profits**
    Even the highest profit doesn't reach the risk free rate 3.65% vs. 3.99%. Profits are mainly negative, showing that the Box spreads, found with the      Box Spread scanner built earlier, didn't succeed at all in locking in the risk free rate.

**No “free lunch”**  
   Commissions, slippage and financing (repo) costs will fully erode these few‐percent raw returns, leaving no true arbitrage.

**Bottom line**  
   After sensible liquidity filters, box spreads lock in only tiny per‑trade profits or losses. The sky‑high annualised numbers are simply the mathematical result of scaling short‑dated, small raw returns to a yearly basis.  

**next step:**  
   - scan **longer‑dated maturities** (1–3 months) where annualisation effects are smaller and yields could cluster around the true risk‐free rate


In [50]:
# --------------------------------------------------------------------
# Analyze 1–3 month (30–90 day) box‐spread yields vs. the risk‑free rate
# --------------------------------------------------------------------

# Reload the saved results (if not already in memory)
boxes = pd.read_csv("new_all_box_spreads.csv")

# Define the mid‑term window (30 to 90 days to expiry)
mid_term = boxes[(boxes['days2exp'] >= 30) & (boxes['days2exp'] <= 90)]

# Show summary statistics
rf_annual = 0.0399     # e.g. 3.99% 1‑year Treasury yield
print(f"Risk‑free rate (annual) : {rf_annual:.2%}")
print("Mid‑term box spreads (30–90 days) summary:\n")
print(mid_term['ann_yield'].describe())

# Display top & bottom mid‑term yields
best_mid  = mid_term.sort_values("ann_yield", ascending=False).head(10)
worst_mid = mid_term.sort_values("ann_yield", ascending=True ).head(10)

print("\n=== TOP 10 mid‑term (1–3 mo) annualised yields ===")
display(best_mid[["underlying","expiry","days2exp","K1","K2","ann_yield","raw_return"]])

print("\n=== BOTTOM 10 mid‑term (1–3 mo) annualised yields ===")
display(worst_mid[["underlying","expiry","days2exp","K1","K2","ann_yield","raw_return"]])


Risk‑free rate (annual) : 3.99%
Mid‑term box spreads (30–90 days) summary:

count    10.000000
mean     -0.155175
std       0.284444
min      -0.711039
25%      -0.272824
50%      -0.009386
75%       0.031325
max       0.036542
Name: ann_yield, dtype: float64

=== TOP 10 mid‑term (1–3 mo) annualised yields ===


,underlying,expiry,days2exp,K1,K2,ann_yield,raw_return
58,^VIX,2025-06-18,55,16.0,100.0,0.036542,0.005506
63,^VIX,2025-06-18,55,30.0,100.0,0.035264,0.005314
64,^VIX,2025-06-18,55,31.0,100.0,0.031892,0.004806
61,^VIX,2025-06-18,55,28.0,100.0,0.029627,0.004464
55,^VIX,2025-06-18,55,16.0,28.0,-0.005526,-0.000833
57,^VIX,2025-06-18,55,16.0,31.0,-0.013246,-0.001996
56,^VIX,2025-06-18,55,16.0,30.0,-0.023617,-0.003559
60,^VIX,2025-06-18,55,28.0,31.0,-0.355893,-0.053628
59,^VIX,2025-06-18,55,28.0,30.0,-0.575758,-0.086758
62,^VIX,2025-06-18,55,30.0,31.0,-0.711039,-0.107143



=== BOTTOM 10 mid‑term (1–3 mo) annualised yields ===


,underlying,expiry,days2exp,K1,K2,ann_yield,raw_return
62,^VIX,2025-06-18,55,30.0,31.0,-0.711039,-0.107143
59,^VIX,2025-06-18,55,28.0,30.0,-0.575758,-0.086758
60,^VIX,2025-06-18,55,28.0,31.0,-0.355893,-0.053628
56,^VIX,2025-06-18,55,16.0,30.0,-0.023617,-0.003559
57,^VIX,2025-06-18,55,16.0,31.0,-0.013246,-0.001996
55,^VIX,2025-06-18,55,16.0,28.0,-0.005526,-0.000833
61,^VIX,2025-06-18,55,28.0,100.0,0.029627,0.004464
64,^VIX,2025-06-18,55,31.0,100.0,0.031892,0.004806
63,^VIX,2025-06-18,55,30.0,100.0,0.035264,0.005314
58,^VIX,2025-06-18,55,16.0,100.0,0.036542,0.005506


### Why the mean annualised yield (≈ -15.52 %) is well below the risk‑free rate (≈ 4 %)

**Buying at the ask and selling at the bid**  
   On average, the net entry cost  
   $$
     C_{\text{entry}}
     = \text{ask}_{\text{long legs}}
     \;-\;\text{bid}_{\text{short legs}}
   $$  
   **exceeds** the present‑value of the guaranteed payoff  
   $$
     (K_2 - K_1)\,e^{-rT}.
   $$  
   Market‑makers pocket the difference (the “spread”), so a retail buyer of boxes earns a negative yield on average.

**I used the undiscounted payoff when computing raw profit**  
   $$
     \text{profit} = (K_2 - K_1)\;-\;C_{\text{entry}},
   $$  
   rather than  
   $$
     \underbrace{(K_2 - K_1)\,e^{-rT}}_{\text{PV of payoff}}
     \;-\;C_{\text{entry}}.
   $$  



When one wishes to **infer** the market’s implicit discount rate from a box spread, it is necessary to **solve** for \(r\) rather than plug in an assumed rate up front.  The logic proceeds as follows:

1. **Undiscounted entry cost vs guaranteed payoff**  
   - The entry cost today is:  
     $$
       C_{\rm entry}
       = 
       \bigl(\text{price of long‑call at }K_1\bigr)
       -\bigl(\text{price of short‑call at }K_2\bigr)
       -\bigl(\text{price of short‑put at }K_1\bigr)
       +\bigl(\text{price of long‑put at }K_2\bigr).
     $$
   - At expiry the payoff is certainly $K_2 - K_1$.

2. **Solving for the implied \(r\)**  
   Rather than **discounting** the payoff by a preset rate, set today’s cost equal to the **present value** of the future payoff:
   $$
     C_{\rm entry}
     = (K_2 - K_1)\,e^{-r_{\rm imp}\,T}
     \quad\Longrightarrow\quad
     r_{\rm imp}
     = -\frac{1}{T}\,\ln\!\Bigl(\frac{C_{\rm entry}}{K_2 - K_1}\Bigr).
   $$
   - This rearrangement **extracts** $r_{imp}$ from market quotes.  
   - If instead one coded:
     ```python
     pv_payoff = (K2 - K1) * np.exp(-r_assumed * T)
     profit    = pv_payoff - C_entry
     ```
     a discount rate would have been **pre‑assumed**, removing the ability to infer the market‑implied rate.

---

## Theoretical vs. Empirical Implied Rates

- **The theoretical benchmark**:  
  Using the **1 Yr U.S. Treasury yield** (from the Treasury’s daily yield curve, April 2025, https://home.treasury.gov/resource-center/data-chart-center/interest-rates/TextView?type=daily_treasury_yield_curve&field_tdr_date_value_month=202504) yields  
  $$
    r_{\rm true} \approx 0.0399 \quad(3.99\%\ \text{p.a.}).
  $$

- **The box spreads’ implication**:  
  The Box spread scanner found 76 Box spreads from european option of ^VIX, ^NDX, ^RUT, ^XSP, ^RUT, ^OEX, ^SPX, ^DJX , which resulted in an mean implied risk free rate **scaled on a yearly basis** of:  
  $$
    \bar r_{\rm imp}\approx -4.418510 \quad(-441.85\%\ \text{p.a.})
  $$  
  because:
  - The long legs are **bought** at the **ask**,  
  - The short legs are **sold** at the **bid**,  
  so that  
  $$
    C_{\rm entry}
    > (K_2 - K_1)\,e^{-r_{\rm true}\,T},
  $$
  which when solved for $r$ becomes **negative**.

> **Key takeaway:**  
> One must **solve**  
> $$
   C_{\rm entry} = (K_2 - K_1)\,e^{-r_{imp}\,T}
 $$  
> to back out the market’s discount rate.  
> The difference between $r_{imp}$ and the Treasury yield corresponds exactly to the **round‑trip transaction cost**  (bid–ask spread, stub quotes, limited size, etc.).


**Why I annualize returns as `ann_yld = raw_ret / T_year`**

- **Raw return** (`raw_ret`) measures the payoff (profit) **over the option’s actual life** $T_{year}$ in years.  
- To compare strategies with different maturities on a common basis, I convert this to a **per‑annum** figure.  
- The simplest way is a **linear annualization**:
  $$
    \text{annualized yield}
    \;=\;
    \frac{\text{raw\_return}}{T_{\rm year}},
    \quad
    T_{\rm year} = \frac{\text{days to expiry}}{365}.
  $$


**Deriving the “Linear” Annualization Formula**

Suppose a box‐spread (or any trade) produces a **raw return** \(R\) over its actual life of \(T\) years (e.g. 30 days → $T=30/365$).  We want a **simple‑interest** annual rate \(r\) that would give the same return if it applied uniformly over a full year.

1. **Assume simple interest** (no compounding):  
   - If you earn a rate \(r\) per annum, then over a fraction \(T\) of a year you earn  
     $$
       \text{interest} \;=\; r_{annual} \times T.
     $$
2. **Match to the observed return**:  
   - By definition the trade returned \(R\) over period \(T\).  Under the simple‑interest assumption you require  
     $$
       R = r_{annual} \times T.
     $$
3. **Solve for the annual rate** \(r\):  
   $$
     r_{annual} = \frac{R}{T}.
   $$
4. **Translate to code**:  
   ```python
   # T_year = days_to_expiry / 365
   if T_year > 0 and not np.isnan(raw_ret):
       ann_yld = raw_ret / T_year
   else:
       ann_yld = np.nan
